# **Signal treratment**

Since the dataset is sampled to 2000hrz, we don't need to apply a sampling or an anti-aliasing filter, but in term of computation we can apply another sampling to 500hrz

In [31]:
import numpy as np
import pandas as pd
from Data_viz import data_cleaning, training_folders_names
import os
import soundfile as sf

## **Split the records to 5 seconds to have a numerical presentation**

In [32]:
counts, data = data_cleaning()    

In [33]:
data.columns = ["file", "class", "sure", "folder name"]
data

,file,class,sure,folder name
0,a0001,1,1,training-a
1,a0002,1,1,training-a
2,a0003,1,1,training-a
3,a0004,1,1,training-a
4,a0005,1,1,training-a
...,...,...,...,...
3148,f0110,-1,1,training-f
3149,f0111,-1,1,training-f
3150,f0112,-1,1,training-f
3151,f0113,-1,1,training-f


Listing the wav files

In [34]:
# for index, row in data.iterrows():
base_path = r"C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Dataset"

def split_signals_and_build(data: pd.DataFrame, segment_duration : int = 5, expected_sample_rate : int = 2000):

    segment_size = segment_duration * expected_sample_rate
    
    segments_data = []

    for _, row in data.iterrows():
        path = os.path.join(
            base_path,
            row["folder name"],
            row["file"] + ".wav"
        )

        signal, sample_rate = sf.read(path)

        # Ensure the recording has the expected sampling rate
        if sample_rate != expected_sample_rate:
            raise ValueError(
                f"{path}: expected {expected_sample_rate} Hz, "
                f"got {sample_rate} Hz"
            )


        for start in range(0, len(signal), segment_size):
            segment = signal[start:start + segment_size]

            original_length = len(segment)

            if original_length < segment_size:
                segment = np.pad(
                    segment,
                    (0, segment_size - original_length),
                    mode="constant",
                    constant_values=0
                )

            # Keep only complete 5-second segments
            if len(segment) == segment_size:
                segments_data.append({
                    "file": row["file"],
                    "class": row["class"],
                    "sure": row["sure"],
                    "folder name": row["folder name"],
                    "start_sample": start,
                    "signal": segment
                })

    data_augmented = pd.DataFrame(segments_data)

    return data_augmented


### **First approach : keep the sampling on 2000hrz**

In [35]:
data_augmented = split_signals_and_build(data=data)

,file,class,sure,folder name,start_sample,signal
0,a0001,1,1,training-a,0,"[-0.002105712890625, -0.004119873046875, -0.00..."
1,a0001,1,1,training-a,10000,"[0.00970458984375, 0.012298583984375, 0.013946..."
2,a0001,1,1,training-a,20000,"[0.004638671875, 0.0067138671875, 0.0055541992..."
3,a0001,1,1,training-a,30000,"[-0.010986328125, -0.00982666015625, -0.009796..."
4,a0001,1,1,training-a,40000,"[0.00946044921875, 0.0096435546875, 0.00955200..."
...,...,...,...,...,...,...
14733,f0114,-1,1,training-f,20000,"[0.00958251953125, -0.011962890625, -0.0456542..."
14734,f0114,-1,1,training-f,30000,"[0.045013427734375, 0.033905029296875, 0.01852..."
14735,f0114,-1,1,training-f,40000,"[0.0032958984375, 0.01275634765625, -0.0716552..."
14736,f0114,-1,1,training-f,50000,"[0.002899169921875, 0.04986572265625, 0.089080..."
